In [ ]:
# Rankine-Hugoniot Jump Conditions and Shock Cut (paper figures)
#
# Reads the data cached by notebooks/rh_conds/rh_conds_mime.ipynb and reproduces the
# two figures used in the previous paper's rh_conds.ipynb - "rh" and "shock_cut" - here
# named "rh_mime" and "shock_cut_mime". This notebook has no graphet/TristanRun
# dependency - it only needs the saved NetCDF file.

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12


def save_polished_fig(name, dpi=300, metadata=None):
    plt.savefig(f"/scratch/gpfs/ar3676/plots/{name}", bbox_inches='tight', dpi=dpi, metadata=metadata)


findex = 250  # must match the findex used when the data was saved

data = xr.open_dataset(f"../rh_conds/rh_conds_data_{findex}.nc")

X_CUTOFF = data.attrs['X_CUTOFF']
xlim = data.attrs['xlim']
x_position_of_min_pb = data.attrs['x_position_of_min_pb']
boundary = data.attrs['boundary']

print(f"Loaded RH data for findex={findex}: theta = {data.attrs['angle_in_deg']:.2f} deg "
      f"({data.attrs['theta_method']}), steps = {list(data.attrs['my_steps'])}")


In [ ]:
# Six-Panel RH Plot ("rh_mime")

def plot_squant(axis, quantity):
    right_val = data[quantity].sel(x=X_CUTOFF * xlim, method="nearest")
    axis.plot(data.x, data[quantity] / right_val)


eqn_fsize = 18
AXIS_FONTSIZE = plt.rcParams['font.size'] * 0.7  # tick/axis-label size, 30% smaller than default

fig, axes = plt.subplots(3, 2, figsize=(8, 4), sharex=True)
ax1, ax2, ax3, ax4, ax5, ax6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]
plt.subplots_adjust(hspace=0, wspace=0.2)

# Mass

plot_squant(ax1, 'mass_flux')
ax1.text(0.45, 0.10, r"$\rho v_n$", transform=ax1.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax1.set_ylim(0, 2.2)

# Pressure

plot_squant(ax2, 'pressure')
ax2.set_ylim(0, 1.4)
ax2.text(0.35, 0.10, r"$\rho v_n^2 + P_g + P_B$", transform=ax2.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Energy

plot_squant(ax3, 'energy')
energy_expression = r'$\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}$'
ax3.text(0.15, 0.10, energy_expression, transform=ax3.transAxes, fontsize=12, verticalalignment='bottom')
ax3.set_ylim(0, 4.2)

# B_n

plot_squant(ax4, 'B_n')
ax4.text(0.45, 0.10, '$B_n$', transform=ax4.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax4.set_ylim(0, 2.2)

# Poynting Flux

plot_squant(ax5, 'poynting')
ax5.text(0.35, 0.10, r'$\rho v_n v_t - \frac{B_n B_t} {4 \pi}$', transform=ax5.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax5.set_ylim(0, 2.2)

# Electric Field

plot_squant(ax6, 'E_ideal')
ax6.text(0.35, 0.10, '$B_n v_t - v_n B_t$', transform=ax6.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')
ax6.set_ylim(0, 2.2)

for ax_row in axes:
    for ax in ax_row:
        ax.set_xlim(x_position_of_min_pb, xlim * X_CUTOFF)
        ax.axvline(boundary, color='green', linestyle='--')
        ax.tick_params(axis='both', which='major', labelsize=AXIS_FONTSIZE)

for ax in [ax3, ax6]:
    ax.set_xlabel("$x$ [$c / \omega_p$]", fontsize=AXIS_FONTSIZE)

plt.show()

save_polished_fig("rh_mime")


In [ ]:
# Shock Cut ("shock_cut_mime")

fig, ax = plt.subplots(figsize=(8, 4))

max_jz = data.jz.sel(x=slice(x_position_of_min_pb, xlim)).max()

ax.axvline(boundary, color='green', linestyle='--')
ax.plot(data.x, data.ndens, label="$n / n_0$")
ax.plot(data.x, data.pn, label="$P_g / P_{B0}$")
ax.plot(data.x, data.pb, label="$P_B / P_{B0}$")
ax.plot(data.x, data.jz / max_jz, label=r"$j_z / j_S$")

ax.set_xlim(x_position_of_min_pb, xlim * X_CUTOFF)
ax.set_xlabel("$x$ [$c / \omega_p$]", fontsize=AXIS_FONTSIZE)
ax.tick_params(axis='both', which='major', labelsize=AXIS_FONTSIZE)
ax.legend()

plt.tight_layout()
plt.show()

save_polished_fig("shock_cut_mime")
